# 02 — Tái lập lựa chọn danh mục

**Mục tiêu:** đối chiếu toàn bộ targets và universe audit của năm strategy ở 60 decisions Dev và 24 decisions Holdout.

**Đầu vào:** cùng pin của Notebook 01. **Đầu ra:** bảng kiểm tra, minh họa ứng viên/điểm/target và `02_verification.json`. Không sinh candidate/ranking CSV canonical mới.

`Strategies` và `ranking` chính thức thực hiện tính điểm. Notebook không viết lại thuật toán chọn mã.

### Thiết lập trực tiếp trong notebook

Chọn kernel Python có các phụ thuộc trong lock, rồi **Run All**. Các cell dưới đọc pin và định nghĩa phần điều phối ngay trong notebook; không import helper Python từ `M3`. Logic nghiên cứu vẫn gọi module chính thức trong `src/delta_t1/m3` và hàm metrics của `scripts/run_m3.py`.

In [1]:
"""Điều phối tái lập M3-C; mọi PIT/ranking/backtest gọi module M3 chính thức."""
from __future__ import annotations

from bisect import bisect_right
from collections import Counter, defaultdict
from datetime import datetime, timezone
import hashlib
import html
import importlib.util
import json
import math
import os
from pathlib import Path
import csv
import subprocess
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'M3/sources.json').is_file())
runtime = ROOT / '.tmp/m3c-notebook-runtime'
if runtime.is_dir():
    sys.path.insert(0, str(runtime))
sys.path.insert(0, str(ROOT / 'src'))
from delta_t1.m3 import inputs, ranking, universe
from delta_t1.m3.benchmark import benchmark_nav
from delta_t1.m3.engine import run as portfolio_run
from delta_t1.m3.prices import PriceBook
from delta_t1.m3.strategies import Strategies

PIN = json.loads((ROOT / 'M3/sources.json').read_text(encoding='utf-8'))
spec = importlib.util.spec_from_file_location('m3_official_runner', ROOT / 'scripts/run_m3.py')
official = importlib.util.module_from_spec(spec)
spec.loader.exec_module(official)
NAMES = dict(combined='Combined', market_only='Market-only', financial_only='Financial-only',
             equal_weight='Equal-weight', momentum_only='Momentum-only', vnindex='VNINDEX')
RTOL, ATOL = 1e-9, 1e-10

### Đọc dữ liệu và đối chiếu

Identifiers/schema/order phải khớp chính xác; số thực theo tolerance của evidence.

In [2]:
def sha(path):
    return inputs.sha256(path)

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

def csv_rows(path):
    with Path(path).open(encoding='utf-8-sig', newline='') as file:
        return list(csv.DictReader(file))

def grouped(rows, key):
    result = defaultdict(list)
    for row in rows:
        result[row[key]].append(row)
    return dict(result)

def equal(actual, expected, context=''):
    if isinstance(actual, bool):
        assert str(actual) == str(expected), (context, actual, expected)
    elif isinstance(actual, (float, int)):
        assert math.isfinite(float(expected)) and math.isclose(float(actual), float(expected), rel_tol=RTOL, abs_tol=ATOL), (context, actual, expected)
    elif actual is None:
        assert expected in (None, ''), (context, actual, expected)
    else:
        assert str(actual) == str(expected), (context, actual, expected)

def compare_rows(actual, expected, context):
    assert len(actual) == len(expected), (context, len(actual), len(expected))
    for index, (left, right) in enumerate(zip(actual, expected)):
        assert set(left) == set(right), (context, index, set(left) ^ set(right))
        for key, value in left.items():
            equal(value, right[key], (context, index, key))

def table(rows, columns=None):
    """Bảng HTML hiển thị trong notebook, không sinh canonical CSV trung gian."""
    from IPython.display import HTML, display
    columns = list(columns or (list(rows[0]) if rows else []))
    def cell(value):
        if value is None:
            return ''
        if isinstance(value, float):
            return f'{value:.8g}'
        return str(value)
    head = ''.join(f'<th>{html.escape(c)}</th>' for c in columns)
    body = ''.join('<tr>' + ''.join(f'<td>{html.escape(cell(row.get(c)))}</td>' for c in columns) + '</tr>' for row in rows)
    display(HTML('<div style="overflow:auto"><table style="border-collapse:collapse;font-size:13px">'
                 '<thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table></div>'))

def notebook_code_sha256(path):
    notebook = read_json(path)
    pieces = [''.join(c['source']) for c in notebook['cells'] if c['cell_type'] == 'code']
    return hashlib.sha256('\n\n'.join(pieces).encode('utf-8')).hexdigest()

### Pin nguồn và kiểm tra input

Hash bảy input; lấy trần/sàn trong cùng lượt đọc C8. Nếu các notebook cùng phiên dùng `M3C_OUTPUT_DIR`, proof được kiểm tra fingerprint và tái sử dụng.

In [3]:
def source_identity():
    scope = ['src/delta_t1/m3', 'scripts/run_m3.py', 'configs/experiments/m3_strategy_v1.json']
    assert not subprocess.check_output(['git', 'diff', PIN['source_commit'], '--', *scope], cwd=ROOT), 'M3 source khác commit đã pin'
    assert not subprocess.check_output(['git', 'status', '--porcelain', '--', *scope], cwd=ROOT), 'M3 source/config chưa sạch'
    return dict(source_commit=PIN['source_commit'],
                execution_commit=subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True).strip(),
                branch=subprocess.check_output(['git', 'branch', '--show-current'], cwd=ROOT, text=True).strip(),
                worktree_status=subprocess.check_output(['git', 'status', '--short'], cwd=ROOT, text=True).strip(),
                python=sys.version.split()[0], executable=sys.executable,
                module_hashes={str(p.relative_to(ROOT)).replace('\\', '/'): sha(p)
                               for p in [*sorted((ROOT / 'src/delta_t1/m3').glob('*.py')), ROOT / 'scripts/run_m3.py']})

def input_paths(config):
    return dict(terminal_sha256=inputs.BUNDLE, indicators_sha256=inputs.INDICATORS,
                availability_sha256=inputs.AVAILABILITY, c8_benchmark_sha256=inputs.BENCHMARK,
                c8_calendar_sha256=inputs.CALENDAR, c8_prices_sha256=inputs.PRICES_C8,
                b_pit_labels_sha256=ROOT / config['financial_pit']['artifact'] / 'labels.csv')

def fingerprint(path):
    stat = path.stat()
    return dict(size=stat.st_size, mtime_ns=stat.st_mtime_ns)

def validate_runs():
    config = read_json(ROOT / PIN['config'])
    assert config['status'] == 'LOCKED' and sha(ROOT / PIN['config']) == PIN['config_sha256']
    runs = {}
    for segment, pin in PIN['runs'].items():
        seal_path = ROOT / pin['seal']
        assert sha(seal_path) == pin['seal_sha256'], segment
        seal = read_json(seal_path)
        folder = ROOT / seal['run']
        manifest = read_json(folder / 'manifest.json')
        assert seal['status'] == 'PASS' and len(seal['artifacts_sha256']) == 33
        assert {p.name for p in folder.iterdir() if p.is_file()} == set(seal['artifacts_sha256'])
        for name, digest in seal['artifacts_sha256'].items():
            assert sha(folder / name) == digest, (segment, name)
        for key in ['git_commit', 'config_sha256', 'inputs_sha256']:
            assert manifest[key] == seal[key]
        assert manifest['git_commit'] == PIN['source_commit']
        assert manifest['config_sha256'] == PIN['config_sha256']
        assert manifest['financial_source'] == 'B_PIT' and not manifest['targets_only'] and not manifest['git_dirty']
        assert manifest['segment'] == segment and manifest['status'] == 'LOCKED'
        runs[segment] = dict(folder=folder, manifest=manifest, seal=seal,
                             metrics=read_json(folder / 'metrics.json'))
    assert runs['development']['manifest']['inputs_sha256'] == runs['holdout']['manifest']['inputs_sha256']
    return config, runs

def preflight(output):
    """Hash đủ input; cùng lượt đọc C8 lưu trần/sàn cho hai segment để tránh scan lặp."""
    output = Path(output).resolve()
    output.mkdir(parents=True, exist_ok=True)
    identity = source_identity()
    config, runs = validate_runs()
    expected = runs['development']['manifest']['inputs_sha256']
    dates = {r['date'] for run in runs.values() for r in csv_rows(run['folder'] / 'nav_combined.csv')}
    hashes = {}; limits = []
    paths = input_paths(config)
    for key, path in paths.items():
        before = fingerprint(path)
        if key == 'c8_prices_sha256':
            digest = hashlib.sha256(); seen = set()
            with path.open('rb') as file:
                for line in file:
                    digest.update(line)
                    marker = b'"trade_date": "'
                    pos = line.index(marker) + len(marker)
                    date = line[pos:pos + 10].decode('ascii')
                    if date in dates:
                        row = json.loads(line)
                        pair = row['ticker'], date
                        assert pair not in seen, pair
                        seen.add(pair)
                        limits.append([*pair, row['raw_close'], row['ceiling_price'], row['floor_price']])
            actual = digest.hexdigest()
        else:
            actual = sha(path)
        assert before == fingerprint(path), ('Input bị thay đổi khi đọc', path)
        assert actual == expected[key], key
        hashes[key] = dict(path=str(path.relative_to(ROOT)).replace('\\', '/'), sha256=actual, **before)
    limits_path = output / 'verified_limits.json'
    write_json(limits_path, limits)
    bpit = ROOT / config['financial_pit']['artifact']
    result = dict(status='PASS', created_at=datetime.now(timezone.utc).isoformat(), identity=identity,
                  sources_sha256=sha(ROOT / 'M3/sources.json'), inputs=hashes,
                  bpit_metadata={name: sha(bpit / name) for name in ['manifest.json', 'models.json', 'audit.json']},
                  limits=dict(path=str(limits_path), sha256=sha(limits_path), dates=sorted(dates), rows=len(limits)))
    write_json(output / 'preflight.json', result)
    return result

In [4]:
class Context:

    def __init__(self):
        self.config, self.runs = validate_runs()
        self.identity = source_identity()
        self.output = Path(os.environ.get('M3C_OUTPUT_DIR', str(ROOT / 'M3/reproduction' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')))).resolve()
        self.output.mkdir(parents=True, exist_ok=True)
        proof = self.output / 'preflight.json'
        if not proof.exists():
            preflight(self.output)
        self.proof = read_json(proof)
        assert self.proof['status'] == 'PASS' and self.proof['sources_sha256'] == sha(ROOT / 'M3/sources.json')
        assert self.proof['identity']['module_hashes'] == self.identity['module_hashes']
        for key, path in input_paths(self.config).items():
            record = self.proof['inputs'][key]
            assert fingerprint(path) == {k: record[k] for k in ['size', 'mtime_ns']}, ('Cần chạy lại preflight', path)
            assert record['sha256'] == self.runs['development']['manifest']['inputs_sha256'][key]
        bpit = ROOT / self.config['financial_pit']['artifact']
        for name, digest in self.proof['bpit_metadata'].items():
            assert sha(bpit / name) == digest
        self.market = inputs.market_snapshots()
        self.availability = inputs.availability()
        self.indicators = inputs.indicators()
        self.sectors = inputs.sectors()
        financial, order, _, cutoffs = inputs.bpit(bpit, include_cutoffs=True)
        self.cutoffs = cutoffs
        self.strategies = Strategies(self.config, self.market, financial, self.availability, self.sectors, self.indicators, fin_order=order, financial_cutoffs=cutoffs)
        self.calendar = inputs.calendar()

    def read(self, segment, kind, strategy=None):
        name = f'{kind}_{strategy}.csv' if strategy else f'{kind}.csv'
        return csv_rows(self.runs[segment]['folder'] / name)

    def decisions(self, segment):
        m = self.runs[segment]['manifest']
        return sorted((d for d in self.market if m['segment_start'] <= d <= m['segment_end_date']))

    def execution(self, decision):
        i = bisect_right(self.calendar, decision)
        return self.calendar[i] if i < len(self.calendar) else ''

    def provenance(self):
        return [dict(input=key, **row) for key, row in self.proof['inputs'].items()]

    def target_rows(self, segment, strategy, date, result=None):
        result = result or self.strategies.target_for(strategy, date)
        execution = self.execution(date)
        status = result.get('status', 'OK')
        if not execution or execution > self.runs[segment]['manifest']['segment_end_date']:
            status = 'NOT_EXECUTED_SEGMENT_END'
        target = result.get('target') or {}
        rows = [dict(decision_date=date, execution_date=execution, ticker=t, rank=i + 1, target_weight=target[t], candidates=result['candidates'], status=status) for i, t in enumerate(result['picks'])]
        if not rows:
            rows = [dict(decision_date=date, execution_date=execution, ticker='', rank='', target_weight='', candidates=result['candidates'], status=status)]
        return (rows, result)

    def finish(self):
        for key, path in input_paths(self.config).items():
            record = self.proof['inputs'][key]
            assert fingerprint(path) == {k: record[k] for k in ['size', 'mtime_ns']}, ('Input thay đổi trong phiên', path)
        bpit = ROOT / self.config['financial_pit']['artifact']
        for name, digest in self.proof['bpit_metadata'].items():
            assert sha(bpit / name) == digest
        for segment in self.runs:
            seal = self.runs[segment]['seal']
            for name, digest in seal['artifacts_sha256'].items():
                assert sha(self.runs[segment]['folder'] / name) == digest
        source_identity()
        return 'PASS — nguồn canonical giữ nguyên; không có Final performance.'

In [5]:
ctx = Context()
print('Code nguồn:', ctx.identity['source_commit'])
print('Code thực thi:', ctx.identity['execution_commit'])
print('Python:', ctx.identity['executable'])
print('Config và nguồn canonical: PASS')
print('Evidence:', ctx.output)

Code nguồn: c9c1f5f0c88f2dbb3ebbb50effa47003b24b66cf
Code thực thi: c9c1f5f0c88f2dbb3ebbb50effa47003b24b66cf
Python: C:\Users\HP\Downloads\Phân cụm động lượng TTCK\.tmp\m3c-kernel\Scripts\python.exe
Config và nguồn canonical: PASS
Evidence: C:\Users\HP\Downloads\Phân cụm động lượng TTCK\M3\reproduction\20261104T154826484210Z-direct-notebooks


## 1. Tái lập đủ hai đoạn và năm strategy

Đối chiếu từng trường, từng dòng của target và audit. Identifiers/status/order phải khớp chính xác; số thực dùng rtol=1e-9, atol=1e-10. Một discrepancy làm notebook dừng.

### Tái lập và đối chiếu lựa chọn danh mục

In [6]:
def selection(ctx):
    summary = []
    per_period = []
    for segment in ctx.runs:
        for strategy in ctx.config['strategies']:
            targets = []
            audit = []
            for date in ctx.decisions(segment):
                rows, result = ctx.target_rows(segment, strategy, date)
                targets.extend(rows)
                audit.extend(result['audit'])
                per_period.append(dict(segment=segment, strategy=strategy, decision_date=date, candidates=result['candidates'], N_selected=len(result['picks']), status=rows[0]['status']))
            compare_rows(targets, ctx.read(segment, 'targets', strategy), (segment, strategy, 'targets'))
            compare_rows(audit, ctx.read(segment, 'universe_audit', strategy), (segment, strategy, 'audit'))
            expected = ctx.runs[segment]['manifest']['counts'][strategy]
            records = [r for r in per_period if r['segment'] == segment and r['strategy'] == strategy]
            assert len(records) == expected['decisions']
            assert sum((r['status'] == 'UNIVERSE_TOO_SMALL' for r in records)) == expected['too_small']
            summary.append(dict(segment=segment, strategy=NAMES[strategy], decisions=len(records), targets_rows=len(targets), audit_rows=len(audit), result='PASS'))
    write_json(ctx.output / '02_verification.json', dict(status='PASS', summary=summary, periods=per_period))
    return (summary, per_period)

In [7]:
summary, periods = selection(ctx)
table(summary)
print("Số strategy/segment đã đối chiếu:", len(summary))

segment,strategy,decisions,targets_rows,audit_rows,result
development,Combined,60,787,45646,PASS
development,Market-only,60,1595,45646,PASS
development,Financial-only,60,4012,45646,PASS
development,Equal-weight,60,25647,45646,PASS
development,Momentum-only,60,7722,45646,PASS
holdout,Combined,24,259,30155,PASS
holdout,Market-only,24,828,30155,PASS
holdout,Financial-only,24,1706,30155,PASS
holdout,Equal-weight,24,18087,30155,PASS
holdout,Momentum-only,24,5439,30155,PASS


Số strategy/segment đã đối chiếu: 10


## 2. Tập ứng viên và số mã được chọn của Combined

N_selected là số mã mục tiêu, chưa phải số mã thực giữ sau giao dịch. Kỳ dưới N_min không tạo mua mới. Kỳ cuối vẫn lưu target nhưng không execution ngoài segment.

In [8]:
table([r for r in periods if r["strategy"] == "combined"])

segment,strategy,decision_date,candidates,N_selected,status
development,combined,2019-01-31,63,19,OK
development,combined,2019-02-28,68,21,OK
development,combined,2019-03-29,4,0,UNIVERSE_TOO_SMALL
development,combined,2019-04-26,8,0,UNIVERSE_TOO_SMALL
development,combined,2019-05-31,15,5,OK
development,combined,2019-06-28,15,5,OK
development,combined,2019-07-31,14,5,OK
development,combined,2019-08-30,11,4,OK
development,combined,2019-09-30,22,7,OK
development,combined,2019-10-31,12,4,OK


## 3. Ví dụ sáu tiêu chí, điểm và tỷ trọng mục tiêu

Ba MC: mom_126 cao, downside_vol_63 thấp, liquidity_21 cao. Ba FC: roe/f_score/z_score cao. Score là trung bình percentile theo nhóm, Combined cân bằng hai nhóm 50/50. Thiếu một phần tiêu chí dùng 0,5 và có flag; thiếu toàn bộ FC bị loại trước N_min.

Các bảng sau minh họa kỳ đầu mỗi đoạn, gồm một số mã được chọn và không được chọn. Điểm/số liệu lấy từ kết quả module chính thức. Giá trị tỷ trọng 0,05 nghĩa là 5%.

### Tạo bảng minh họa các tiêu chí và điểm

In [9]:
def ranking_example(ctx, segment, date=None):
    date = date or ctx.decisions(segment)[0]
    result = ctx.strategies.target_for('combined', date)
    candidates, _ = ctx.strategies._universe(date, market_rule=ctx.config['market_rule'])
    raw = ctx.strategies._rows(candidates)
    pct = {}
    for key, higher in {**ctx.config['market_criteria'], **ctx.config['financial_criteria']}.items():
        pct[key] = ranking.percentiles({t: value.get(key) if higher or value.get(key) is None else -value[key] for t, value in raw.items()})
    picks = {t: i + 1 for i, t in enumerate(result['picks'])}
    rows = []
    for ticker, score in result.get('scores', {}).items():
        for group, names in [('s_market', ctx.config['market_criteria']), ('s_fin', ctx.config['financial_criteria'])]:
            values = [pct[k][ticker] if pct[k][ticker] is not None else 0.5 for k in names]
            equal(sum(values) / len(values), score[group], ('display percentile', ticker, group))
        row = dict(ticker=ticker, **raw[ticker])
        row.update({f'pct_{k}': v[ticker] if v[ticker] is not None else 0.5 for k, v in pct.items()})
        row.update(S_market=score['s_market'], S_fin=score['s_fin'], combined_score=score['score'], rank=picks.get(ticker), selected=ticker in picks, target_weight=result.get('target', {}).get(ticker) if result.get('target') else None, flags=';'.join(score['missing']))
        rows.append(row)
    rows.sort(key=lambda r: (r['rank'] is None, r['rank'] or 0, -r['combined_score'], r['ticker']))
    return rows

In [10]:
for segment in ctx.runs:
    example = ranking_example(ctx, segment)
    visible = [r for r in example if r["selected"]][:8] + [r for r in example if not r["selected"]][:5]
    print(segment, ctx.decisions(segment)[0])
    table(visible, ["ticker", "mom_126", "downside_vol_63", "liquidity_21", "roe", "f_score", "z_score", "flags"])
    table(visible, ["ticker", "pct_mom_126", "pct_downside_vol_63", "pct_liquidity_21", "pct_roe", "pct_f_score", "pct_z_score", "S_market", "S_fin", "combined_score", "rank", "selected", "target_weight"])

development 2019-01-31


ticker,mom_126,downside_vol_63,liquidity_21,roe,f_score,z_score,flags
NNC,0.20327787,0.10159131,6.102381e+08,0.43610991,,10.092908,f_score
DSN,0.17873738,0.084510229,3.3975238e+08,0.39839753,,46.409378,f_score
LIX,0.2109317,0.11393366,1.1654471e+09,0.31983437,,6.7237653,f_score
HMC,0.28863615,0.10642615,1.5159433e+09,0.22074554,,3.6236556,f_score
AMV,1.3486909,0.25329453,1.9747315e+09,0.24568886,,6.2754029,f_score
DHA,0.157327,0.13234853,1.0369524e+09,0.16819737,,14.187729,f_score
TV2,0.19210424,0.21059127,3.2842502e+09,0.53323178,6,2.8490179,
GHC,0.18713532,0.15274029,84931357,0.36065311,,7.4399273,f_score
CLC,0.085318782,0.18127127,1.6247619e+08,0.25758886,,5.3700387,f_score
CMX,2.3877523,0.44212041,1.6314762e+09,0.64766686,,1.0595794,f_score


ticker,pct_mom_126,pct_downside_vol_63,pct_liquidity_21,pct_roe,pct_f_score,pct_z_score,S_market,S_fin,combined_score,rank,selected,target_weight
NNC,0.75806452,0.93548387,0.64516129,0.90322581,0.5,0.93333333,0.77956989,0.77885305,0.77921147,1,True,0.052631579
DSN,0.69354839,0.98387097,0.58064516,0.85483871,0.5,0.98333333,0.75268817,0.77939068,0.76603943,2,True,0.052631579
LIX,0.79032258,0.88709677,0.74193548,0.75806452,0.5,0.78333333,0.80645161,0.68046595,0.74345878,3,True,0.052631579
HMC,0.87096774,0.90322581,0.79032258,0.4516129,0.5,0.5,0.85483871,0.48387097,0.66935484,4,True,0.052631579
AMV,0.98387097,0.25806452,0.83870968,0.61290323,0.5,0.76666667,0.69354839,0.6265233,0.66003584,5,True,0.052631579
DHA,0.66129032,0.79032258,0.72580645,0.29032258,0.5,0.95,0.72580645,0.58010753,0.65295699,6,True,0.052631579
TV2,0.74193548,0.37096774,0.90322581,0.9516129,0.64285714,0.28333333,0.67204301,0.62593446,0.64898874,7,True,0.052631579
GHC,0.70967742,0.67741935,0.29032258,0.80645161,0.5,0.85,0.55913978,0.7188172,0.63897849,8,True,0.052631579
CLC,0.61290323,0.5,0.41935484,0.66129032,0.5,0.68333333,0.51075269,0.61487455,0.56281362,,False,
CMX,1,0,0.82258065,1,0.5,0.033333333,0.60752688,0.51111111,0.559319,,False,


holdout 2024-01-31


ticker,mom_126,downside_vol_63,liquidity_21,roe,f_score,z_score,flags
DGC,0.21997259,0.14048383,1.8726952e+11,0.67340848,6,9.1547368,
PNJ,0.13740329,0.087001772,1.2427302e+11,0.25049814,6,6.7512795,
DCM,0.045300418,0.16316189,8.5289143e+10,0.47881048,9,5.6000712,
FPT,0.15678697,0.11305484,2.0520051e+11,0.27240118,6,3.6659892,
CSV,0.4506719,0.16919445,2.6593429e+10,0.28639736,,5.9278086,f_score
LIX,0.29724754,0.1013767,2.9167143e+09,0.26696007,7,5.6980092,
FOX,0.41488885,0.08954352,4.5323661e+09,0.31606823,6,2.5111635,
BMP,0.23680404,0.20790781,1.7840363e+10,0.28252206,9,10.800512,
DDV,-0.0027700531,0.15794763,7.3923548e+09,0.2193572,,5.4429052,f_score
SBH,0.66475505,0.56373339,1.4184445e+08,0.30202581,,17.438265,f_score


ticker,pct_mom_126,pct_downside_vol_63,pct_liquidity_21,pct_roe,pct_f_score,pct_z_score,S_market,S_fin,combined_score,rank,selected,target_weight
DGC,0.59375,0.73958333,0.89583333,0.96875,0.58441558,0.94791667,0.74305556,0.83369408,0.78837482,1,True,0.033333333
PNJ,0.48958333,0.96875,0.875,0.75,0.58441558,0.92708333,0.77777778,0.75383297,0.76580538,2,True,0.033333333
DCM,0.30208333,0.59375,0.78125,0.94791667,0.94155844,0.86458333,0.55902778,0.91801948,0.73852363,3,True,0.033333333
FPT,0.51041667,0.88541667,0.90625,0.79166667,0.58441558,0.6875,0.76736111,0.68786075,0.72761093,4,True,0.033333333
CSV,0.875,0.54166667,0.65625,0.84375,0.5,0.90625,0.69097222,0.75,0.72048611,5,True,0.033333333
LIX,0.65625,0.92708333,0.34375,0.78125,0.71428571,0.875,0.64236111,0.79017857,0.71626984,6,True,0.033333333
FOX,0.84375,0.95833333,0.42708333,0.875,0.58441558,0.57291667,0.74305556,0.67744408,0.71024982,7,True,0.033333333
BMP,0.60416667,0.34375,0.58333333,0.8125,0.94155844,0.96875,0.51041667,0.90760281,0.70900974,8,True,0.033333333
DDV,0.25,0.61458333,0.5,0.69791667,0.5,0.82291667,0.45486111,0.67361111,0.56423611,,False,
SBH,0.92708333,0.020833333,0.083333333,0.86458333,0.5,0.98958333,0.34375,0.78472222,0.56423611,,False,


### Bảy kiểm tra phản ví dụ

Giữ các assertion của bộ kiểm tra trước đây: tolerance, NaN, missing/zero, schema/order/status, sai config/seal và đúng population percentile trước loại toàn bộ FC thiếu.

In [11]:
import copy
from types import SimpleNamespace
import unittest
from unittest.mock import patch
class ComparisonGates(unittest.TestCase):
    def test_float_tolerance_accepts_roundoff_but_rejects_material_nav_error(self):
        equal(1.23456789, '1.2345678900000002')
        with self.assertRaises(AssertionError):
            equal(1.23456789, '1.2346')

    def test_nonfinite_values_cannot_pass(self):
        for value in ['nan', 'inf', '-inf']:
            with self.assertRaises(AssertionError):
                equal(1., value)

    def test_missing_is_distinct_from_zero(self):
        equal(None, '')
        with self.assertRaises(AssertionError):
            equal(None, 0)

    def test_shifted_rows_missing_columns_and_status_changes_fail(self):
        actual = [dict(ticker='AAA', weight=.5, status='EXECUTED'), dict(ticker='BBB', weight=.5, status='SCALED')]
        for bad in [list(reversed(actual)), [dict(ticker='AAA', weight=.5), actual[1]],
                    [dict(actual[0], status='BUY_FAILED'), actual[1]]]:
            with self.assertRaises(AssertionError):
                compare_rows(actual, bad, 'negative control')

    def test_bad_pinned_seal_is_rejected(self):
        pins = copy.deepcopy(PIN)
        pins['runs']['development']['seal_sha256'] = '0' * 64
        with patch.dict(PIN, pins, clear=True), self.assertRaises(AssertionError):
            validate_runs()

    def test_bad_config_pin_is_rejected(self):
        pins = copy.deepcopy(PIN)
        pins['config_sha256'] = '0' * 64
        with patch.dict(PIN, pins, clear=True), self.assertRaises(AssertionError):
            validate_runs()

    def test_display_percentile_retains_pre_exclusion_population(self):
        raw = {'AAA': {'mom': 1., 'roe': 2.}, 'BBB': {'mom': 2., 'roe': 1.},
               'CCC': {'mom': 3., 'roe': None}}
        scores = ranking.combined_scores(raw, {'mom': True}, {'roe': True})
        result = dict(scores=scores, picks=['AAA'], target={'AAA': 1.})
        context = object.__new__(Context)
        context.config = dict(market_criteria={'mom': True}, financial_criteria={'roe': True}, market_rule='leading')
        context.strategies = SimpleNamespace(target_for=lambda *a: result,
                                             _universe=lambda *a, **kw: ([], []),
                                             _rows=lambda *a: raw)
        rows = ranking_example(context, 'development', '2020-01-31')
        by_ticker = {row['ticker']: row for row in rows}
        self.assertEqual(set(by_ticker), {'AAA', 'BBB'})
        self.assertEqual(by_ticker['BBB']['pct_mom'], .5)
        self.assertEqual(by_ticker['BBB']['S_market'], .5)
suite = unittest.defaultTestLoader.loadTestsFromTestCase(ComparisonGates)
negative_controls = unittest.TextTestRunner(verbosity=1).run(suite)
assert negative_controls.wasSuccessful() and negative_controls.testsRun == 7
write_json(ctx.output / 'negative_controls.json', dict(status='PASS', tests=negative_controls.testsRun))

.

.

.

.

.

.

.

----------------------------------------------------------------------
Ran 7 tests in 0.012s

OK


## 4. Kết luận

Target khớp không đồng nghĩa lệnh mua thành công. Bước execution và phần danh mục thực giữ được kiểm chứng ở Notebook 03. Không chọn lại rule từ lợi suất Dev/Holdout.

In [12]:
print(ctx.finish())
print("NOTEBOOK_02_PASS")

PASS — nguồn canonical giữ nguyên; không có Final performance.
NOTEBOOK_02_PASS


In [13]:
write_json(ctx.output / 'direct_02_done.json', dict(status='PASS', notebook='02_Tai_Lap_Lua_Chon_Danh_Muc.ipynb', code_sha256=notebook_code_sha256(ROOT / 'M3/notebooks/02_Tai_Lap_Lua_Chon_Danh_Muc.ipynb')))
print('DIRECT_NOTEBOOK_02_PASS')

DIRECT_NOTEBOOK_02_PASS
